In [ ]:
from Tools_U1_plaquette import *

In [ ]:
# -------- Hamiltonian parameters --------
m_list = [10, 20, 50, 100]
g = 1.0
lam = 1.0
E_g = g / 4
E_J = lam
neig = 9
Nsize = 25

# -------- Storage --------
E_P_all = {}
E_eff_all = {}
err_all = {}
n12_exp_all = {}
n24_exp_all = {}
n13_exp_all = {}
n34_exp_all = {}
n1_exp_all = {}
n2_exp_all = {}
n3_exp_all = {}
n4_exp_all = {}

# -------- Operators --------
if Nsize % 2 == 0:
    M = int(Nsize / 2) 
    n_c = sp.spdiags(np.arange(-M, M), [0], 2*M, 2*M)
    id_c = sp.eye(2 * M )
else:
    M = (Nsize-1) // 2 
    n_c = sp.spdiags(np.arange(-M, M + 1), [0], 2 * M + 1, 2 * M + 1)
    id_c = sp.eye(2 * M + 1)
###### CHOICE OF ORDER HERE: [ θ12, θ24, θ13, θ34 ]
n12_op = kron_n(n_c, id_c, id_c, id_c)
n24_op = kron_n(id_c, n_c, id_c, id_c)     
n13_op = kron_n(id_c, id_c, n_c, id_c)
n34_op = kron_n(id_c, id_c, id_c, n_c)

# -------- Loop --------
for m in m_list:
    print(f"Computing for m={m}...")
    E_m = m / 4

    # -------- Full Hamiltonian --------
    H_P = inf_space_LGT_plaquette_charge_basis_Matrix_reduced4vars_alphas0(
        EC_g1=E_g, EJ1=E_J,
        EC_g2=E_g, EJ2=E_J,
        EC_g3=E_g, EJ3=E_J,
        EC_g4=E_g, EJ4=E_J,
        Nsize=Nsize,
        EC_m1=E_m, EC_m2=E_m, EC_m3=E_m, EC_m4=E_m
    )
    off_diag = (H_P - sp.diags(H_P.diagonal(), format="csr")).nnz
    if off_diag == 0:
        H_P = H_P + sp.eye(Nsize**4)
    print("H_P constructed.")
    E_P, V_P = spla.eigsh(
        H_P, k=neig, which="SA",
        tol=1e-10, maxiter=10000
    )
    if off_diag == 0:
        E_P -= 1.0
    E_P -= E_P[0]
    E_P_all[m] = E_P
    print("H_P diagonalized.")

    # -------- Expectation values --------
    n12_vals = np.zeros(neig)
    n24_vals = np.zeros(neig)
    n13_vals = np.zeros(neig)
    n34_vals = np.zeros(neig)
    n1_vals = np.zeros(neig)
    n2_vals = np.zeros(neig)
    n3_vals = np.zeros(neig)
    n4_vals = np.zeros(neig)

    for alpha in range(neig):
        psi = V_P[:, alpha]
        n12_vals[alpha] = np.real(np.vdot(psi, n12_op @ psi))
        n24_vals[alpha] = np.real(np.vdot(psi, n24_op @ psi))
        n13_vals[alpha] = np.real(np.vdot(psi, n13_op @ psi))
        n34_vals[alpha] = np.real(np.vdot(psi, n34_op @ psi))
        n1_vals[alpha] = n12_vals[alpha].copy() + n13_vals[alpha].copy()
        n2_vals[alpha] = n24_vals[alpha].copy() - n12_vals[alpha].copy()
        n3_vals[alpha] = n34_vals[alpha].copy() - n13_vals[alpha].copy()
        n4_vals[alpha] = - n24_vals[alpha].copy() - n34_vals[alpha].copy()

    n12_exp_all[m] = n12_vals
    n24_exp_all[m] = n24_vals
    n13_exp_all[m] = n13_vals
    n34_exp_all[m] = n34_vals
    n1_exp_all[m] = n1_vals
    n2_exp_all[m] = n2_vals
    n3_exp_all[m] = n3_vals
    n4_exp_all[m] = n4_vals
    print("Expectation values computed.")

    # -------- Effective Hamiltonian --------
    H_eff = inf_space_LGT_plaquette_charge_basis_Matrix_reduced4vars_alphas0_effective_staticm(
        EC_g1=E_g, EC_g2=E_g, EC_g3=E_g, EC_g4=E_g,
        EC_m=E_m, EJ=E_J,
        Nsize=Nsize
    )
    off_diag_eff = (H_eff - sp.diags(H_eff.diagonal(), format="csr")).nnz
    if off_diag_eff == 0:
        H_eff = H_eff + sp.eye(Nsize)
    print("H_eff constructed.")
    E_eff, _ = spla.eigsh(
        H_eff, k=neig, which="SA",
        tol=1e-10, maxiter=10000
    )
    if off_diag_eff == 0:
        E_eff -= 1.0
    E_eff -= E_eff[0]
    E_eff_all[m] = E_eff
    print("H_eff diagonalized.")

    # -------- Relative error --------
    Diff = np.abs(E_eff - E_P)
    err = Diff / (np.abs(E_P) + 1e-12)
    err_all[m] = err

In [ ]:
filename_Heff_varying_m = f"H_P_vs_H_eff_g_{int(g)}_lam_{int(lam)}_Nsize_{int(Nsize)}_neig_{int(neig)}_varying_m_with_expval_ns_def.npz"

E_P_all_arr = np.stack([E_P_all[m] for m in m_list])
E_eff_all_arr = np.stack([E_eff_all[m] for m in m_list])
err_all_arr = np.stack([err_all[m] for m in m_list])
n12_arr = np.stack([n12_exp_all[m] for m in m_list])
n24_arr = np.stack([n24_exp_all[m] for m in m_list])
n13_arr = np.stack([n13_exp_all[m] for m in m_list])
n34_arr = np.stack([n34_exp_all[m] for m in m_list])
n1_arr = np.stack([n1_exp_all[m] for m in m_list])
n2_arr = np.stack([n2_exp_all[m] for m in m_list])
n3_arr = np.stack([n3_exp_all[m] for m in m_list])
n4_arr = np.stack([n4_exp_all[m] for m in m_list])

np.savez_compressed(
    filename_Heff_varying_m,
    m = np.array(m_list),
    E_P_all=E_P_all_arr,
    E_eff_all=E_eff_all_arr,
    err_all=err_all_arr,
    n12_exp_all=n12_arr,
    n24_exp_all=n24_arr,
    n13_exp_all=n13_arr,
    n34_exp_all=n34_arr, 
    n1_exp_all=n1_arr,
    n2_exp_all=n2_arr,
    n3_exp_all=n3_arr,
    n4_exp_all=n4_arr
)
print("Saved data for varying m in ", filename_Heff_varying_m)